# ProjectEcho — Phase 20: PPO Training

**Setup:** In Kaggle, attach the `projectecho-training` dataset (your uploaded `training/` folder) as input data before running.

Trained checkpoints and TensorBoard logs are written to `/kaggle/working/checkpoints/`.
Download `best_model.zip` or `final_model.zip` from the Output tab when done.

In [ ]:
# Install training dependencies (Kaggle already has torch; this just adds SB3)
import subprocess, sys
subprocess.run([
    sys.executable, "-m", "pip", "install", "--quiet",
    "stable-baselines3>=2.3.0",
    "sb3-contrib>=2.3.0",
    "gymnasium>=0.29.0",
], check=True)

In [ ]:
# Make the training/ source files importable
import sys, os

# Adjust this path to match your Kaggle dataset slug
TRAINING_DIR = "/kaggle/input/projectecho-training"
if TRAINING_DIR not in sys.path:
    sys.path.insert(0, TRAINING_DIR)

OUTPUT_DIR = "/kaggle/working/checkpoints"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("sys.path OK. Output dir:", OUTPUT_DIR)

In [ ]:
# Quick smoke-test: make sure the env imports and resets without error
from env_wrappers import LeagueEnv
env = LeagueEnv(num_tribes=4, max_ticks=50)
obs, _ = env.reset(seed=42)
print(f"Observation shape: {obs.shape}  (expected ({51},))")
mask = env.action_masks()
print(f"Action mask (valid actions): {mask.sum()} / 12")

In [ ]:
# Run training.
#
# Kaggle free tier: 2 CPU cores, 13 GB RAM, 30 GPU hrs/week.
# --envs 4 is safe; drop to 2 if you hit OOM.
# --timesteps 500000 takes ~2-3 hours on P100 GPU.
#
# Resume a previous run by adding:
#   --resume /kaggle/working/checkpoints/best_model

result = subprocess.run([
    sys.executable,
    os.path.join(TRAINING_DIR, "train.py"),
    "--timesteps", "500000",
    "--envs",      "4",
    "--seed",      "42",
    "--output-dir", OUTPUT_DIR,
    "--win-rate",  "0.60",
], check=False)

print("Exit code:", result.returncode)

In [ ]:
# List saved checkpoints
for f in sorted(os.listdir(OUTPUT_DIR)):
    path = os.path.join(OUTPUT_DIR, f)
    size = os.path.getsize(path) // 1024
    print(f"{f:40s}  {size} KB")